### Install our package

In [0]:
!pip install /Workspace/Users/balazs.balogh@cubixedu.com/cubix_data_engineer_capstone-0.2.24-py3-none-any.whl

### Create the Catalogs

In [0]:
%skip

spark.sql("DROP CATALOG capstone CASCADE")

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS source_system")
spark.sql("USE CATALOG source_system")
spark.sql("CREATE SCHEMA IF NOT EXISTS source_system")
spark.sql("CREATE VOLUME IF NOT EXISTS source_system.source_system.source_files")

spark.sql("CREATE CATALOG IF NOT EXISTS capstone")
spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS capstone.bronze.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

source_system_tables = [
    "calendar",
    "customers",
    "product_category",
    "product_subcategory",
    "products",
    "sales"
]

for table in source_system_tables:
    dbutils.fs.mkdirs(f"/Volumes/source_system/source_system/source_files/{table}")
    dbutils.fs.mkdirs(f"/Volumes/capstone/bronze/bronze/{table}")

In [0]:
display(dbutils.fs.ls("/Volumes/source_system/source_system/source_files/product_category"))

### Read file from Volume test

In [0]:
%skip

from pyspark.sql import DataFrame, SparkSession

def read_file_from_volume(full_path: str, format: str) -> DataFrame:
    """Reads a file from UC Volume and returns it as a Spark DataFrame.

    :param file_path:       The path to the file in the data lake.
    :param format:          The format of the file ("csv", "json", "delta", "parquet").
    :return:                DataFrame with a loaded data.
    """

    if format not in ["csv", "parquet", "delta"]:
        raise ValueError(f"Invalid format: {format}. Supported formats are: csv, json, parquet, delta.")

    spark = SparkSession.getActiveSession()
    if not spark:
        raise RuntimeError("No active SparkSession found.")

    reader = spark.read.format(format)
    if format == "csv":
        reader = reader.option("header", "true")

    return reader.load(full_path)

In [0]:
from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume

calendar_source_df = read_file_from_volume(full_path="/Volumes/source_system/source_system/source_files/calendar/calendar.csv", format="csv")
display(calendar_source_df)

### Write file to Volume test

In [0]:
from cubix_data_engineer_capstone.utils.databricks import write_file_to_volume

calendar_source_df = write_file_to_volume(
    df=calendar_source_df,
    full_path="/Volumes/capstone/bronze/bronze/calendar/calendar.csv", 
    format="csv"
)

### Bronze ingestion test

In [0]:
from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest_volume

source_path = "/Volumes/source_system/source_system/source_files/calendar/"
bronze_path = "/Volumes/capstone/bronze/bronze/calendar/"

bronze_ingest_volume(
    source_path=source_path,
    bronze_path=bronze_path,
    file_name="calendar.csv"
)

In [0]:
from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume

calendar_bronze_df = read_file_from_volume(full_path=bronze_path+"calendar.csv", format="csv")
display(calendar_bronze_df)

### Prepare the SCD 1 logic

In [0]:
from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest_volume

datasets = {
    "calendar": {
        "file_name": "calendar.csv",
    },
    "customers": {
        "file_name": "customers_1.csv",
    },
    "product_category": {
        "file_name": "product_category.csv",
    },
    "product_subcategory": {
        "file_name": "product_subcategory.csv",
    },
    "products": {
        "file_name": "products.csv",
    },
    "sales": {
        "file_name": "sales_1.csv",
    },
}

for dataset_key, params in datasets.items():

    bronze_ingest_volume(
        source_path=f"/Volumes/source_system/source_system/source_files/{dataset_key}",
        bronze_path=f"/Volumes/capstone/bronze/bronze/{dataset_key}",
        file_name=params["file_name"],
    )

    print(f"{dataset_key} ({params['file_name']}) ingested")

In [0]:
from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume

raw_product_subcategory = read_file_from_volume(
    full_path="/Volumes/capstone/bronze/bronze/product_subcategory/product_subcategory.csv",
    format="csv"
)
display(raw_product_subcategory)

In [0]:
from cubix_data_engineer_capstone.etl.silver.product_subcategory import get_product_subcategory

product_subcategory = get_product_subcategory(raw_product_subcategory)
display(product_subcategory)

In [0]:
(
    product_subcategory
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.silver.product_subcategory")
)

In [0]:
%sql 

SELECT * FROM capstone.silver.product_subcategory LIMIT 5

In [0]:
import pyspark.sql.functions as sf
import pyspark.sql.types as st

schema = st.StructType([
    st.StructField("ProductSubcategoryKey", st.IntegerType(), True),
    st.StructField("ProductCategoryKey", st.IntegerType(), True),
    st.StructField("EnglishProductSubCategoryName", st.StringType(), True),
    st.StructField("SpanishProductSubCategoryName", st.StringType(), True),
    st.StructField("FrenchProductSubCategoryName", st.StringType(), True),
])

data = [
    (
        1,
        1,
        "Test", # was "Mountain Bikes"
        "Bicicleta de montaña",
        "VTT"
    ),
    (
        99,
        99,
        "Test",
        "Test",
        "Test"
    )
]

new_product_subcategory_data = spark.createDataFrame(data, schema)
display(new_product_subcategory_data)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession

def scd1_uc(spark: SparkSession, table_name: str, new_data: DataFrame, primary_key: str):
    """
    Slowly Changing Dimension Type 1 for UC Volumes.
    Compares the master Delta table with new data, updating or inserting as needed.

    :param spark:       SparkSession.
    :param table_name:  Name of the table.
    :param new_data:    DataFrame with the new data.
    :param primary_key: Column name used as primary key.
    """
    delta_master = DeltaTable.forName(spark, table_name)

    (
        delta_master
        .alias("master")
        .merge(
            new_data.alias("new_data"),
            f"master.{primary_key} = new_data.{primary_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
scd1_uc(
    spark,
    "capstone.silver.product_subcategory",
    new_product_subcategory_data,
    "ProductSubcategoryKey"
)

In [0]:
%sql

SELECT * FROM capstone.silver.product_subcategory

In [0]:
%sql

DESCRIBE HISTORY capstone.silver.product_subcategory

### Create the base Data Warehouse on UC (whole process)

In [0]:
from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest_volume

from cubix_data_engineer_capstone.etl.silver.calendar import get_calendar
from cubix_data_engineer_capstone.etl.silver.customers import get_customers
from cubix_data_engineer_capstone.etl.silver.products import get_products
from cubix_data_engineer_capstone.etl.silver.product_subcategory import get_product_subcategory
from cubix_data_engineer_capstone.etl.silver.product_category import get_product_category
from cubix_data_engineer_capstone.etl.silver.sales import get_sales 

from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume

In [0]:
spark.sql("DROP CATALOG capstone CASCADE")

spark.sql("CREATE CATALOG IF NOT EXISTS capstone")
spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS capstone.bronze.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

source_system_tables = [
    "calendar",
    "customers",
    "product_category",
    "product_subcategory",
    "products",
    "sales"
]

for table in source_system_tables:
    dbutils.fs.mkdirs(f"/Volumes/capstone/bronze/bronze/{table}")

In [0]:
datasets = {
    "calendar": {
        "file_name": "calendar.csv",
        "function": get_calendar
    },
    "customers": {
        "file_name": "customers_1.csv",
        "function": get_customers
    },
    "product_category": {
        "file_name": "product_category.csv",
        "function": get_product_category
    },
    "product_subcategory": {
        "file_name": "product_subcategory.csv",
        "function": get_product_subcategory
    },
    "products": {
        "file_name": "products.csv",
         "function": get_products
    },
    "sales": {
        "file_name": "sales_1.csv",
        "function": get_sales
    },
}

for dataset, params in datasets.items():

    bronze_ingest_volume(
            source_path=f"/Volumes/source_system/source_system/source_files/{dataset}",
            bronze_path=f"/Volumes/capstone/bronze/bronze/{dataset}",
            file_name=params["file_name"],
        )
    
    print(f"{dataset} has been processed in the Bronze layer.")

    raw_dataframe = read_file_from_volume(
        full_path=f"/Volumes/capstone/bronze/bronze/{dataset}/{params['file_name']}",
        format="csv"
    )

    transform_func = params["function"]
    transformed_dataframe = transform_func(raw_dataframe)

    (
        transformed_dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"capstone.silver.{dataset}")
    )

    print(f"{dataset} has been processed in the Silver layer.")

In [0]:
%sql

SELECT * FROM capstone.silver.sales

In [0]:
%sql

SELECT * FROM capstone.silver.customers

### Gold layer

In [0]:
from cubix_data_engineer_capstone.etl.gold.wide_sales import get_wide_sales
from cubix_data_engineer_capstone.etl.gold.daily_product_category_metrics import get_daily_product_category_metrics
from cubix_data_engineer_capstone.etl.gold.daily_sales_metrics import get_daily_sales_metrics

In [0]:
calendar_master = spark.table("capstone.silver.calendar")
customers_master = spark.table("capstone.silver.customers")
product_subcategory_master = spark.table("capstone.silver.product_subcategory")
product_category_master = spark.table("capstone.silver.product_category")
products_master = spark.table("capstone.silver.products")
sales_master = spark.table("capstone.silver.sales")

#### Wide Sales

In [0]:
wide_sales_df = get_wide_sales(
    sales_master=sales_master,
    calendar_master=calendar_master,
    customers_master=customers_master,
    products_master=products_master,
    product_subcategory_master=product_subcategory_master,
    product_category_master=product_category_master
)

In [0]:
wide_sales_df.count()

In [0]:
display(wide_sales_df)

In [0]:
(
    wide_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.wide_sales")
)

In [0]:
%sql

-- Total sales and profit by month to identify high-performing months.
SELECT 
    MonthName, 
    CalendarYear,
    SUM(SalesAmount) AS TotalSales, 
    SUM(Profit) AS TotalProfit
FROM 
    capstone.gold.wide_sales
GROUP BY 
    CalendarYear, 
    MonthName, 
    MonthNumberOfYear
ORDER BY 
    CalendarYear, 
    MonthNumberOfYear

In [0]:
%sql

-- Sales by customer demographics
SELECT 
    MaritalStatus, 
    Gender, 
    AVG(SalesAmount) AS AvgSales, 
    COUNT(SalesOrderNumber) AS TotalOrders
FROM 
    capstone.gold.wide_sales
GROUP BY 
    MaritalStatus,
    Gender
ORDER BY 
    AvgSales DESC;

In [0]:
%sql

-- Which products are most often part of high-value orders.
SELECT 
    ProductName, 
    COUNT(SalesOrderNumber) AS HighValueOrderCount
FROM 
    capstone.gold.wide_sales
WHERE 
    HighValueOrder = true
GROUP BY 
    ProductName
ORDER BY 
    HighValueOrderCount DESC;

#### Daily Product Category Metrics

In [0]:
daily_product_category_metrics = get_daily_product_category_metrics(wide_sales_df) 

In [0]:
display(daily_product_category_metrics)

In [0]:
(
    daily_product_category_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_product_category_metrics")
)

#### Daily Sales Metrics

In [0]:
daily_sales_metrics = get_daily_sales_metrics(wide_sales_df) 

In [0]:
display(daily_sales_metrics)

In [ ]:
(
    daily_sales_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_sales_metrics")
)

### Data Quality with The Great Expectations

In [0]:
!pip install great_expectations

In [0]:
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine.sparkdf_execution_engine import SparkDFExecutionEngine
from great_expectations import get_context

In [0]:
# 1. Create a context:
context = get_context()

# 2. Create a Spark Execution Engine
execution_engine = SparkDFExecutionEngine(persist=False)

# 3. Create a Batch from the DataFrame
batch = Batch(data=wide_sales_df)

# 4. Create a Validator with the batch and execution engine
validator = Validator(execution_engine=execution_engine, batches=[batch])

# 5. Add expectations
validator.expect_column_values_to_not_be_null(
    column="SalesOrderNumber", 
)

validator.expect_column_values_to_be_in_set(
    column="Gender", 
    value_set=["Male", "Female"])

validator.expect_column_values_to_be_between(
    column="BirthDate", 
    min_value="1999-01-01",
    max_value=None
)

# 6. Run validation and get results
results = validator.validate()

# 7. Process results
if results["success"]:
    print("All validations passed!")
else:
    print("Some validations failed.")
    for result in results["results"]:
        print(f"Expectation: {result['expectation_config']['type']}")
        print(f"Success: {result['success']}")
        if not result["success"]:
            print(f"Details: {result['result']}")


In [0]:
validation_results = results["results"]

results_data = [
    {
        "Expectation": res["expectation_config"]["type"],
        "Column": res["expectation_config"]["kwargs"].get("column"),
        "Success": res["success"],
        "Count": res["result"].get("element_count", "N/A"),
        "Failed_Records_Count": res["result"].get("unexpected_count", "N/A"),
        "Failed_Records_Percentage": res["result"].get("unexpected_percent", "N/A"),
    }
    for res in validation_results
]

validation_results_df = spark.createDataFrame(results_data)

display(validation_results_df)

(
    validation_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.wide_sales_validation_results")
)